##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time


In [3]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
Train shape: (50000, 32, 32, 3)
Test shape: (10000, 32, 32, 3)


In [4]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])


In [5]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # freeze backbone


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)  # logits
], name="cifar10_efficientnet")

model.summary()


Model: "cifar10_efficientnet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [7]:

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Frozen test accuracy:", test_acc)


Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 115s 145ms/step - accuracy: 0.6437 - loss: 1.0881 - val_accuracy: 0.8826 - val_loss: 0.3612
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 99s 141ms/step - accuracy: 0.7831 - loss: 0.6394 - val_accuracy: 0.8842 - val_loss: 0.3297
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 99s 141ms/step - accuracy: 0.8016 - loss: 0.5806 - val_accuracy: 0.8928 - val_loss: 0.3117
313/313 ━━━━━━━━━━━━━━━━━━━━ 22s 68ms/step - accuracy: 0.8952 - loss: 0.3242
Frozen test accuracy: 0.8932999968528748


In [8]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


history_ft = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = model.evaluate(x_test, y_test)
print("Fine-tuned test accuracy:", test_acc_ft)
print("Fine-tuned avg time per epoch:", sum(time_cb_ft.times)/len(time_cb_ft.times))


Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 131s 169ms/step - accuracy: 0.7553 - loss: 0.7373 - val_accuracy: 0.8900 - val_loss: 0.3472
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 116s 165ms/step - accuracy: 0.8037 - loss: 0.5819 - val_accuracy: 0.8982 - val_loss: 0.3085
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 116s 165ms/step - accuracy: 0.8146 - loss: 0.5388 - val_accuracy: 0.9022 - val_loss: 0.2887
313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - accuracy: 0.9000 - loss: 0.2930
Fine-tuned test accuracy: 0.9004999995231628


NameError: name 'time_cb_ft' is not defined

Which model achieved the highest accuracy?

The ResNet50V2 (fine-tuned) achieved the highest test accuracy.

ResNet50V2 (fine-tuned): 0.9162

EfficientNetB0 (fine-tuned): 0.9005

EfficientNetB0 (frozen): 0.8933

Basic CNN: 0.7028


Which model trained faster?

EfficientNetB0 trained faster than ResNet50V2.
ResNet took about 215 seconds per epoch, while EfficientNet took about 121 seconds per epoch.



How might the architecture explain the differences?

ResNet50V2 is deeper and has more computations, which leads to higher accuracy but slower training.
EfficientNetB0 is designed to be computationally efficient, so it trains faster while still maintaining strong performance.